# Sprint3 Part1 — 하이퍼파라미터 튜닝
**목표:** GridSearchCV / RandomizedSearchCV / Bayesian Optimization으로 최적 하이퍼파라미터 찾기  
**내용:**  
1) 기본 모델 성능을 기준선(Baseline)으로 잡는다  
2) GridSearchCV로 **작은 탐색 공간을 “완전탐색”** 해본다  
3) RandomizedSearchCV로 **큰 탐색 공간을 “확률적 탐색”** 한다  
4) **Bayesian Optimization 기반 탐색**을 수행하고, Best 모델을 재학습한다  

> 이 노트북은 **회귀(Regression)모델** 기준 예시이며, 데이터/피처셋만 바꾸면 그대로 재사용 가능.  
> 아래 코드는 `X_tr_EDA, X_va_EDA, X_te_EDA, y_train, y_val` 가  
이미 만들어져 있다고 가정하고 있으므로, 데이터셋을 코랩에 업로드 하고 시작한다.  


## 0. 환경 준비

In [ ]:
!pip -q install hyperopt

In [ ]:
# 준비된 데이터셋 코랩 업로드 후 로드
import pandas as pd

X_tr_EDA = pd.read_csv('/content/X_tr_EDA.csv')
X_va_EDA = pd.read_csv('/content/X_va_EDA.csv')
y_train = pd.read_csv('/content/y_train.csv')
y_val = pd.read_csv('/content/y_val.csv')

## 1. 공통 설정: 평가 지표/교차검증 함수

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score, make_scorer

import warnings
warnings.filterwarnings("ignore")

# rmse 계산 함수
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

rmse_scorer = make_scorer(rmse, greater_is_better=False)

# CV 교차 검증
cv = KFold(n_splits=5, shuffle=True, random_state=42)

def eval_regression(model, X_tr, y_tr, X_va, y_va, name="model"):
    model.fit(X_tr, y_tr)
    pred_tr = model.predict(X_tr)
    pred_va = model.predict(X_va)

    out = {
        "model": name,
        "RMSE_train": rmse(y_tr, pred_tr),
        "RMSE_val": rmse(y_va, pred_va),
        "R2_train": r2_score(y_tr, pred_tr),
        "R2_val": r2_score(y_va, pred_va),
    }
    return out

## 2. Baseline 모델 (튜닝 전 기준선)

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Baseline (train, val 데이터셋 과적합 확인)
dt_base = DecisionTreeRegressor(random_state=42)

rf_base = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

xgb_base = XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1)

results = []
results.append(eval_regression(dt_base, X_tr_EDA, y_train, X_va_EDA, y_val, "DT_baseline"))
results.append(eval_regression(rf_base, X_tr_EDA, y_train, X_va_EDA, y_val, "RF_baseline"))
results.append(eval_regression(xgb_base, X_tr_EDA, y_train, X_va_EDA, y_val, "XGB_baseline"))

pd.DataFrame(results).sort_values("RMSE_val")

## 3. GridSearchCV (완전탐색) — **DecisionTreeRegressor** 예시
GridSearch는 **탐색 공간이 작을 때** 좋다.  
- 장점: 빠짐 없이 전부 탐색(재현성/설명 용이)  
- 단점: 조합이 커지면 시간 폭발

> 팁: DT는 `max_depth`, `min_samples_leaf`만 잘 잡아도 과적합이 크게 줄어든다.


In [ ]:
from sklearn.model_selection import GridSearchCV

dt = DecisionTreeRegressor(random_state=42)

param_grid_dt = {
    "max_depth": [3, 5, 8, 12, 16],
    "min_samples_split": [2, 10, 20],
    "min_samples_leaf": [1, 3, 8, 15],
    "max_features": [0.6, 0.8, 1.0]}

gs_dt = GridSearchCV(
    estimator=dt,
    param_grid=param_grid_dt,
    scoring=rmse_scorer,
    cv=cv,
    n_jobs=-1,
    verbose=1,
    refit=True)

gs_dt.fit(X_tr_EDA, y_train)

print("Best CV score (neg RMSE):", gs_dt.best_score_)
print("Best params:", gs_dt.best_params_)

best_dt = gs_dt.best_estimator_
pd.DataFrame([eval_regression(best_dt, X_tr_EDA, y_train, X_va_EDA, y_val, "DT_GridSearch_best")])

## 4. RandomizedSearchCV (확률적 탐색) — **RandomForestRegressor** 예시
RandomizedSearch는 **탐색 공간이 큰 모델(RF/XGB 등)** 에서 효율적
- 장점: 같은 시간 대비 더 넓은 공간 탐색  
- 단점: “운”이 작용할 수 있음(시드 고정으로 완화)

> 팁: RF는 `max_features`, `max_depth`, `min_samples_leaf`, `max_samples`를 주로 튜닝


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

rf = RandomForestRegressor(random_state=42, n_jobs=-1)

param_dist_rf = {
    "n_estimators": randint(400, 1200),
    "max_depth": randint(6, 20),
    "min_samples_split": randint(2, 20),
    "min_samples_leaf": randint(1, 10),
    "max_features": uniform(0.4,0.6),
    "max_samples": uniform(0.4, 0.6)}

rs_rf = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist_rf,
    n_iter=10,               # 몇 번 탐색할지 선택 (30~100 사이 추천)
    scoring=rmse_scorer,
    cv=cv,
    n_jobs=-1,
    random_state=42,
    verbose=1,
    refit=True)

rs_rf.fit(X_tr_EDA, y_train)

print("Best CV score (neg RMSE):", rs_rf.best_score_)
print("Best params:", rs_rf.best_params_)

best_rf = rs_rf.best_estimator_
pd.DataFrame([eval_regression(best_rf, X_tr_EDA, y_train, X_va_EDA, y_val, "RF_RandomSearch_best")])

## 5. Bayesian Optimization — **XGBRegressor** 예시
Bayesian Optimization은 Trial 결과를 바탕으로 **“가능성 높은 방향으로”** 탐색해 나가는 방식이라,  
랜덤 탐색보다 **더 적은 시도로** 좋은 파라미터를 찾는 경우가 많다.


### 5-1 Search Space 정의

In [ ]:
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

space = {
    "n_estimators": hp.quniform("n_estimators", 500, 3000, 100), # 간격 설정
    "learning_rate": hp.loguniform("learning_rate", np.log(0.01), np.log(0.2)), # 작은값들을 더 많이 나오게
    "max_depth": hp.quniform("max_depth", 3, 18, 1),
    "min_child_weight": hp.quniform("min_child_weight", 1, 12, 1),
    "subsample": hp.uniform("subsample", 0.4, 1.0),
    "colsample_bytree": hp.uniform("colsample_bytree", 0.4, 1.0),
    "reg_alpha": hp.loguniform("reg_alpha", np.log(0.0001), np.log(1.0)),
    "reg_lambda": hp.loguniform("reg_lambda", np.log(1.0), np.log(12.0)),
}

In [ ]:
def objective(params):

    params["n_estimators"] = int(params["n_estimators"])
    params["max_depth"] = int(params["max_depth"])
    params["min_child_weight"] = int(params["min_child_weight"])

    model = XGBRegressor(
        random_state=42,
        n_jobs=-1,
        early_stopping_rounds=150,
        tree_method="hist",
        **params)

    model.fit(X_tr_EDA, y_train,
              eval_set=[(X_va_EDA, y_val)],
              verbose=False)
    pred = model.predict(X_va_EDA)

    rmse_val = rmse(y_val, pred)

    return {"loss": rmse_val, "status": STATUS_OK}

In [ ]:
trials = Trials()

best = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,
    max_evals=10,            # 마찬가지로 30 ~ 100 사이를 추천
    trials=trials,
    rstate=np.random.default_rng(42)
)

best

In [ ]:
best_params = {
    "n_estimators": int(best["n_estimators"]),
    "learning_rate": float(best["learning_rate"]),
    "max_depth": int(best["max_depth"]),
    "min_child_weight": int(best["min_child_weight"]),
    "subsample": float(best["subsample"]),
    "colsample_bytree": float(best["colsample_bytree"]),
    "reg_alpha": float(best["reg_alpha"]),
    "reg_lambda": float(best["reg_lambda"]),
}

best_params

In [ ]:
best_xgb_bayes = XGBRegressor(
    random_state=42,
    n_jobs=-1,
    tree_method="hist",
    **best_params
)

res_bayes_xgb = eval_regression(
    best_xgb_bayes,
    X_tr_EDA, y_train,
    X_va_EDA, y_val,
    name="XGB_bayes_hyperopt"
)

res_bayes_xgb

## 6. 최종 비교표 (Baseline vs 튜닝)

In [ ]:
# 앞에서 best_dt, best_rf, best_xgb_optuna(또는 best_xgb_cv)가 만들어졌다는 가정
compare = []

compare.append(eval_regression(dt_base, X_tr_EDA, y_train, X_va_EDA, y_val, "DT_baseline"))
compare.append(eval_regression(best_dt, X_tr_EDA, y_train, X_va_EDA, y_val, "DT_Grid_best"))

compare.append(eval_regression(rf_base, X_tr_EDA, y_train, X_va_EDA, y_val, "RF_baseline"))
compare.append(eval_regression(best_rf, X_tr_EDA, y_train, X_va_EDA, y_val, "RF_Random_best"))

compare.append(eval_regression(xgb_base, X_tr_EDA, y_train, X_va_EDA, y_val, "XGB_baseline"))
compare.append(eval_regression(best_xgb_bayes, X_tr_EDA, y_train, X_va_EDA, y_val, "XGB_bayes_hyperopt"))

pd.DataFrame(compare).sort_values("RMSE_val")

## 7. (제출) Test set 최종 평가
- 가장 좋은 모델을 선택하여 predict 추출, [kaggle](https://www.kaggle.com/t/2e8aa09a9f8740dfb3de0472326bf152)에 제출

In [ ]:
X_te_EDA = pd.read_csv('/content/X_te_EDA.csv')
xgb_submission = pd.read_csv('https://raw.githubusercontent.com/rngus4656/datasets/refs/heads/main/S2/sample_submission.csv')

y_test_pred = best_xgb_bayes.predict(X_te_EDA)
xgb_submission['Rented Bike Count'] = y_test_pred
xgb_submission.to_csv('xgb_bayes_submission.csv', index=False)

## 8. 체크 포인트
### 1) 영향을 가장 많이 주고있는 피쳐는?

### 2) 시간대비 가장 효율이 좋았던 탐색 방법은?

### 3) 예측 성능이 가장 잘 나온 모델은?


|index|model|RMSE\_train|RMSE\_val|R2\_train|R2\_val|
|---|---|---|---|---|---|
|5|XGB\_bayes\_hyperopt|0\.7801575843254122|144\.17363527835107|0\.9999985694885254|0\.949095606803894|
|4|XGB\_baseline|59\.60133874400914|164\.90266423522394|0\.991653323173523|0\.9334053993225098|
|2|RF\_baseline|65\.4496052906775|177\.47266909044546|0\.9899349640295501|0\.9228658526711733|
|3|RF\_Random\_best|131\.38568990721353|188\.39643394113557|0\.9594400510670006|0\.9130781267486641|
|1|DT\_Grid\_best|126\.61030565340818|235\.68286472169893|0\.9623348776762112|0\.8639684213866183|
|0|DT\_baseline|0\.0|246\.2205398088151|1\.0|0\.8515321978603599|

## 튜닝 탐색 핵심 리마인드
- **GridSearchCV**: “일정 범위를 완전탐색” (설명/재현에 좋음)
- **RandomizedSearchCV**: “큰 범위를 확률적으로” (시간 대비 효율)
- **BayesianOptimization**: “이전 trial 결과를 보고, 가능성 높은 쪽을 더 탐색”
